# ADEPT trait extraction evaluation
Reproducible evaluation of the final trait matrix on **50 development descriptions covering 49 taxa**. These descriptions informed pipeline refinement; this is not an independent held-out test.

Run cells in order. Requires Python 3.9+ and pandas (`pip install pandas`). Extract the supplied bundle and open this notebook inside its folder. Update the paths below to evaluate other files.

Primary metrics are value-level **precision, recall and F1**, not classification accuracy: there is no defined universe of absent trait values from which to count true negatives. Per-field exact-set agreement is provided separately, both over all scored records and over records with at least one reference or predicted value. All-empty matches can inflate the former.

The included revised reference treats every blank as no relevant information, as agreed for the manuscript. Four Gymnadenia conopsea measurement cells were previously marked unreadable; their predictions now count as false positives. If they remain unknown rather than absent, explicitly exclude them using EXCLUDED_CELLS below instead of treating them as negative annotations.

In [1]:
from pathlib import Path
import re
import json
import hashlib
import platform
import math
import pandas as pd


In [10]:
REFERENCE_PATH = Path('/Users/ben/Projects/ADEPT/Data/adept_trait_extraction_reference.csv')

PREDICTIONS_PATH = Path('/Users/ben/Projects/ADEPT/Code/PrePub/ADEPT/adept/data/processing/output/bhl/angiosperm-peatland-386-385.bhl.traits.xlsx')

METADATA_COLUMNS = {"taxon", "source", "source_id", "matched_name", "key"}

EXCLUDED_CELLS = set()

OUTPUT_DIR = Path(".")

In [11]:
ref_df = pd.read_csv(REFERENCE_PATH)

In [12]:
pred_df_list = []

for source, group_df in ref_df.groupby("source"):
    df = pd.read_excel(PREDICTIONS_PATH, sheet_name=source)

    df["taxon"] = df["taxon"].astype("string").str.lower()


    if source == 'bhl':

        matched_df = df.merge(
            group_df[["taxon", "source_id"]].drop_duplicates(),
            on=["taxon", "source_id"],
            how="inner"
        )

    else:

        matched_df = df[df.taxon.isin(group_df.taxon)]

    pred_df_list.append(matched_df)

pred_df = pd.concat(pred_df_list, ignore_index=True)

# pred_df.taxon

pred_df

,taxon,life form,habitat,habit,clonality,perennial organ,plant min. height [m],plant max. height [m],indumentum,spinescence,...,seed max. length [mm],seed min. diameter [mm],seed max. diameter [mm],dispersal mode,ploidy level (2n),root min. depth [cm],root max. depth [cm],source,source_id,matched_name
0,mentha aquatica,perennial,terrestrial,erect leafy,"solitary plant, suckering",stem,NaN,NaN,pubescent,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,22883976,Mentha aquatica; Mentha sativa
1,plantago lanceolata,perennial,terrestrial,erect leafy,rhizomatous,rhizome,NaN,NaN,NaN,NaN,...,4 mm,NaN,NaN,NaN,"2n = 12, 24",NaN,NaN,bhl,7934546,Plantago lanceolata
2,rumex conglomeratus,NaN,NaN,NaN,NaN,NaN,0.025 m,0.05 m,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,46694846,Rumex conglomeratus
3,ajuga reptans,perennial,terrestrial,"erect leafy/tussock, erect leafy","rhizome, solitary plant, stoloniferous","stem, stolon, rhizome",0.10 m,0.30 m,"glabrous, glabrescent",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ecoflora,1510010040,NaN
4,alopecurus alpinus,perennial,terrestrial,"tussock, scrambler, erect leafy/tussock","rhizomatous, rhizome, stoloniferous","stolon, rhizome",0.10 m,0.45 m,"glabrous, scabrous",NaN,...,NaN,NaN,NaN,ectozoochory,NaN,NaN,NaN,ecoflora,1930940070,NaN
5,ammophila arenaria,perennial,terrestrial,"scrambler, erect leafy/tussock","rhizome, stoloniferous","stolon, rhizome",0.50 m,1.20 m,pubescent,NaN,...,NaN,NaN,NaN,ectozoochory,NaN,NaN,NaN,ecoflora,1930900010,NaN
6,bartsia alpina,perennial,terrestrial,erect leafy,solitary plant,stem,0.10 m,0.25 m,"glandular, pubescent",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ecoflora,1540310010,NaN
7,campanula rotundifolia,perennial,terrestrial,"geoxylices, erect leafy/tussock, erect leafy","rhizome, solitary plant","rootstock, stem, rhizome",0.15 m,0.60 m,NaN,NaN,...,NaN,NaN,NaN,NaN,2n=68,NaN,NaN,ecoflora,1680011410,NaN
8,chrysosplenium oppositifolium,perennial,terrestrial,"herb, erect leafy",solitary plant,stem,0.05 m,0.15 m,"glabrous, hairy",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ecoflora,730030030,NaN
9,dactylorhiza maculata,perennial,terrestrial,"herb, erect leafy","tuber, solitary plant","stem, tuber",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ecoflora,2030190110,NaN


In [28]:
def get_row_key(row):
    parts = [row.taxon, row.source]
    if row.source == 'bhl':
        source_id = int(row.source_id)
        parts.append(str(source_id))

    
    return '|'.join(parts)

ref_df['key'] = ref_df.apply(get_row_key,  axis=1)    
pred_df['key'] = pred_df.apply(get_row_key,  axis=1)  


## Load and validate alignment
Join on the supplied unique `key`, never row order. Keys identify the target taxon and source, plus BHL page where applicable. `matched_name` is provenance, not a trait. Missing/extra keys, duplicated keys or mismatched trait schemas stop evaluation rather than silently dropping records. Column names are kept intact: `seeds per fruit min` and `seeds per fruit min.` remain separate to reproduce the manuscript analysis.

In [30]:

for label, frame in [("reference", ref_df), ("predictions", pred_df)]:
    if "key" not in frame or frame["key"].isna().any():
        raise ValueError(f"{label}: missing record keys")
    if frame["key"].duplicated().any():
        raise ValueError(f"{label}: duplicate keys")
missing = set(ref_df["key"]) - set(pred_df["key"])
extra = set(pred_df["key"]) - set(ref_df["key"])
if missing or extra:
    raise ValueError(f"Unmatched keys: missing predictions={missing}, extra predictions={extra}")
fields = [c for c in ref_df.columns if c not in METADATA_COLUMNS]
prediction_fields = {c for c in pred_df.columns if c not in METADATA_COLUMNS}
if set(fields) != prediction_fields:
    raise ValueError(f"Trait schema mismatch: {set(fields) ^ prediction_fields}")
reference = ref_df.set_index("key")
predictions = pred_df.set_index("key").loc[reference.index]
for column in ["taxon", "source"]:
    if not reference[column].equals(predictions[column]):
        raise ValueError(f"Aligned {column} values disagree")
valid_cells = {(key, field) for key in reference.index for field in fields}
if EXCLUDED_CELLS - valid_cells:
    raise ValueError("An exclusion does not identify an existing cell")
print(f"{len(reference)} aligned records; {reference['taxon'].nunique()} taxa; {len(fields)} trait fields")
print(reference["source"].value_counts().to_string())

50 aligned records; 49 taxa; 80 trait fields
source
efloras.flora_of_north_america    15
efloras.flora_of_china            12
ecoflora                          12
efloras.flora_of_pakistan          8
bhl                                3


## Matching rules
Missing values become empty sets. Lowercase text, trim surrounding whitespace, split comma-separated entries and deduplicate. Do not stem, merge synonyms, use fuzzy matching or apply numeric tolerance. Measurement fields (with units in square brackets) normalise scalar numeric formatting and remove a matching unit suffix. A conflicting unit raises an error; this notebook does not silently convert units. Strip the leading `2n =` notation in the ploidy field.

TP = intersection; FP = predicted minus reference; FN = reference minus predicted. Undefined ratios are NaN (shown as NA in exports). F1 is calculated directly as 2TP/(2TP+FP+FN); it is zero for a nonempty field with no matches, and undefined for an entirely empty field.

In [31]:
def normalise(value, field):
    if pd.isna(value):
        return set()
    text = str(value).lower().strip()
    if "[" in field:
        expected = re.search(r"\[([^]]+)\]", field).group(1)
        suffix = re.search(r"(cm|mm|m)\s*$", text)
        if suffix and suffix.group(1) != expected:
            raise ValueError(f"Unit mismatch in {field}: {value!r}")
        text = re.sub(r"\s*(cm|mm|m)\s*$", "", text)
        try:
            number = float(text)
        except ValueError:
            pass
        else:
            if not math.isfinite(number):
                raise ValueError(f"Non-finite measurement: {value!r}")
            return {str(number)}
        if text and not re.fullmatch(r"[0-9.,+\-–−()\s]+", text):
            raise ValueError(f"Non-numeric measurement or annotation note in {field}: {value!r}")
    if field == "ploidy level (2n)":
        text = re.sub(r"^2n\s*=\s*", "", text)
    return {item.strip() for item in text.split(",") if item.strip()}

def ratio(numerator, denominator):
    return numerator / denominator if denominator else float("nan")

def metrics(tp, fp, fn):
    return {"precision": ratio(tp, tp + fp), "recall": ratio(tp, tp + fn),
            "f1": ratio(2 * tp, 2 * tp + fp + fn)}

In [32]:
cell_rows = []
for key in reference.index:
    for field in fields:
        if (key, field) in EXCLUDED_CELLS:
            continue
        gold = normalise(reference.at[key, field], field)
        predicted = normalise(predictions.at[key, field], field)
        cell_rows.append({
            "key": key, "taxon": reference.at[key, "taxon"],
            "source": reference.at[key, "source"], "field": field,
            "tp": len(gold & predicted), "fp": len(predicted - gold),
            "fn": len(gold - predicted), "reference_values": len(gold),
            "predicted_values": len(predicted), "reference_present": bool(gold),
            "predicted_present": bool(predicted), "active": bool(gold | predicted),
            "exact_match": gold == predicted,
            "reference_normalised": json.dumps(sorted(gold)),
            "predicted_normalised": json.dumps(sorted(predicted)),
            "extra_values": json.dumps(sorted(predicted - gold)),
            "missed_values": json.dumps(sorted(gold - predicted)),
        })
cell_scores = pd.DataFrame(cell_rows)
field_rows = []
for field in fields:
    part = cell_scores[cell_scores["field"] == field]
    tp, fp, fn = [int(part[c].sum()) for c in ["tp", "fp", "fn"]]
    active = part[part["active"]]
    field_rows.append({
        "field": field, "reference_values": int(part["reference_values"].sum()),
        "predicted_values": int(part["predicted_values"].sum()),
        "reference_records": int(part["reference_present"].sum()),
        "predicted_records": int(part["predicted_present"].sum()),
        "tp": tp, "fp": fp, "fn": fn, **metrics(tp, fp, fn),
        "scored_records": len(part), "active_records": len(active),
        "exact_matches_all": int(part["exact_match"].sum()),
        "exact_matches_active": int(active["exact_match"].sum()),
        "exact_match_rate_all": ratio(int(part["exact_match"].sum()), len(part)),
        "exact_match_rate_active": ratio(int(active["exact_match"].sum()), len(active)),
    })
per_field = pd.DataFrame(field_rows)
tp, fp, fn = [int(cell_scores[c].sum()) for c in ["tp", "fp", "fn"]]
overall = pd.DataFrame([{
    "averaging": "micro", "records": len(reference), "taxa": reference["taxon"].nunique(),
    "fields": len(fields), "excluded_cells": len(EXCLUDED_CELLS),
    "reference_values": tp + fn, "predicted_values": tp + fp,
    "tp": tp, "fp": fp, "fn": fn, **metrics(tp, fp, fn)
}])
assert tp + fn == int(per_field["reference_values"].sum())
assert tp + fp == int(per_field["predicted_values"].sum())
print(overall.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\nPer-field results (all fields, including empty fields):")
print(per_field[["field", "reference_values", "tp", "fp", "fn", "precision", "recall", "f1"]].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

averaging  records  taxa  fields  excluded_cells  reference_values  predicted_values   tp  fp  fn  precision   recall       f1
    micro       50    49      80               0              1299              1194 1081 113 218   0.905360 0.832179 0.867228

Per-field results (all fields, including empty fields):
                        field  reference_values  tp  fp  fn  precision  recall    f1
                    life form                36  36   2   0      0.947   1.000 0.973
                      habitat                47  47   2   0      0.959   1.000 0.979
                        habit                94  93   6   1      0.939   0.989 0.964
                    clonality                77  74   3   3      0.961   0.961 0.961
              perennial organ                75  75   3   0      0.962   1.000 0.980
        plant min. height [m]                35  28   0   7      1.000   0.800 0.889
        plant max. height [m]                36  28   0   8      1.000   0.778 0.875
         

## Supplementary table and diagnostic exports
The main supplementary table includes all fields, reference and predicted value counts, TP/FP/FN, precision, recall and F1. Support counts are values, not independent descriptions; see the full per-field CSV for record counts. The LaTeX file is a multipage `longtable`: add `\usepackage{longtable}` to Overleaf's preamble and include the exported file with `\input{supplementary_per_trait.tex}`.

Expected manuscript totals for the supplied files and no exclusions: **TP 1,081; FP 113; FN 218; precision 0.905360; recall 0.832179; micro-F1 0.867228**. No assertion forces these values when using new data. An optional exclusion of all four previously unreadable cells gives TP 1,081; FP 109; FN 218.

`discrepancies.csv` identifies exact extra/missed values for manual review; it does not infer their causes.

In [33]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
overall.to_csv(OUTPUT_DIR / "overall_metrics.csv", index=False, na_rep="NA")
per_field.to_csv(OUTPUT_DIR / "per_field_metrics.csv", index=False, na_rep="NA")
cell_scores.to_csv(OUTPUT_DIR / "cell_level_scores.csv", index=False, na_rep="NA")
cell_scores.loc[(cell_scores["fp"] + cell_scores["fn"]) > 0].to_csv(
    OUTPUT_DIR / "discrepancies.csv", index=False, na_rep="NA")
columns = ["field", "reference_values", "predicted_values", "tp", "fp", "fn", "precision", "recall", "f1"]
supplement = per_field[columns].rename(columns={
    "field": "Trait field", "reference_values": "Reference values", "predicted_values": "Predicted values",
    "tp": "TP", "fp": "FP", "fn": "FN", "precision": "Precision", "recall": "Recall", "f1": "F1"
})
supplement.to_csv(OUTPUT_DIR / "supplementary_per_trait.csv", index=False, float_format="%.3f", na_rep="NA")
def tex_escape(value):
    mapping = {"&": r"\&", "%": r"\%", "_": r"\_", "#": r"\#", "$": r"\$", "{": r"\{", "}": r"\}"}
    return "".join(mapping.get(char, char) for char in str(value))
header = r"Trait field & Ref. & Pred. & TP & FP & FN & P & R & F1 \\"
lines = [r"\begingroup", r"\small", r"\setlength{\tabcolsep}{3pt}",
         r"\begin{longtable}{p{0.32\textwidth}rrrrrrrr}",
         r"\caption{Per-trait extraction performance on the ADEPT development reference. Ref. and Pred. count distinct values within description--trait pairs. P: precision; R: recall. NA denotes an undefined metric.}\label{tab:adept-per-trait}\\",
         r"\hline", header, r"\hline", r"\endfirsthead", r"\hline", header, r"\hline", r"\endhead",
         r"\hline", r"\endfoot"]
for _, row in per_field.iterrows():
    values = [tex_escape(row["field"])] + [str(int(row[c])) for c in columns[1:6]]
    values += ["NA" if pd.isna(row[c]) else f"{row[c]:.3f}" for c in columns[6:]]
    lines.append(" & ".join(values) + r" \\")
lines += [r"\end{longtable}", r"\endgroup"]
(OUTPUT_DIR / "supplementary_per_trait.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
manifest = {
    "reference_sha256": hashlib.sha256(REFERENCE_PATH.read_bytes()).hexdigest(),
    "predictions_sha256": hashlib.sha256(PREDICTIONS_PATH.read_bytes()).hexdigest(),
    "python": platform.python_version(), "pandas": pd.__version__,
    "excluded_cells": sorted(EXCLUDED_CELLS), "trait_fields": fields,
    "matching": "Exact normalised set matching; no synonym mapping or numeric tolerance",
    "blank_reference": "No relevant information; empty set",
    "sample": "Development descriptions, not an independent held-out test",
}
(OUTPUT_DIR / "evaluation_settings.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Exported:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path)

Exported:
.DS_Store
.ipynb_checkpoints
Untitled.ipynb
adept_evaluation
adept_evaluation.ipynb
cell_level_scores.csv
discrepancies.csv
evaluation_settings.json
overall_metrics.csv
per_field_metrics.csv
supplementary_per_trait.csv
supplementary_per_trait.tex
